In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

import seaborn as sns

import matplotlib.dates as mdates

In [ ]:
%config InlineBackend.print_figure_kwargs = {'dpi': 110, 'bbox_inches': 'tight'}

In [ ]:
os.makedirs("../data/original/Rhineland_Palatinate", exist_ok=True)
os.makedirs("../data/preprocessed", exist_ok=True)

## Prevalence data

In [ ]:
df = pd.read_csv("https://static-content.springer.com/esm/art%3A10.1038%2Fs41598-024-64864-1/MediaObjects/41598_2024_64864_MOESM1_ESM.csv", sep=";")
df["Date of SARS-CoV-2 test"] = pd.to_datetime(df["Date of SARS-CoV-2 test"])
df["prevalence"] = df["Positive test"]/df["Valid test"]*100
df.sort_values("Town", inplace=True)

In [ ]:
df["Date of SARS-CoV-2 test"].min()

In [ ]:
df["Date of SARS-CoV-2 test"].max() - df["Date of SARS-CoV-2 test"].min()

In [ ]:
df.groupby("Town").size()

In [ ]:
fig, axs = plt.subplots(nrows=2, figsize=(9, 5), dpi=300, sharex=True)
sns.lineplot(data=df, x="Date of SARS-CoV-2 test", y="prevalence", hue="Town", ax=axs[0])
sns.scatterplot(data=df, x="Date of SARS-CoV-2 test", y="prevalence", hue="Town", ax=axs[0], legend=False)
axs[0].set_ylabel("Prevalence [%]")
sns.lineplot(data=df, x="Date of SARS-CoV-2 test", y="Valid test", hue="Town", ax=axs[1])
sns.scatterplot(data=df, x="Date of SARS-CoV-2 test", y="Valid test", hue="Town", legend=False, ax=axs[1])
axs[1].set_ylabel("Tests [#]")
axs[1].legend_.remove()


for i, ax in enumerate(axs):
    ax = plt.gca()
    axs[i].xaxis.set_major_formatter(mdates.DateFormatter('%y-%m'))
    #axs[i].xaxis.set_major_locator(mdates.MonthLocator(bymonth=(1, 3, 5, 7, 9)))

plt.tight_layout()


In [ ]:
df_sub = df.loc[df["Valid test"]>350]
fig, axs = plt.subplots(nrows=2, figsize=(9, 5), dpi=300, sharex=True)
sns.lineplot(data=df_sub, x="Date of SARS-CoV-2 test", y="prevalence", hue="Town", ax=axs[0])
sns.scatterplot(data=df_sub, x="Date of SARS-CoV-2 test", y="prevalence", hue="Town", ax=axs[0], legend=False)
axs[0].set_ylabel("Prevalence [%]")
sns.lineplot(data=df_sub, x="Date of SARS-CoV-2 test", y="Valid test", hue="Town", ax=axs[1])
sns.scatterplot(data=df_sub, x="Date of SARS-CoV-2 test", y="Valid test", hue="Town", legend=False, ax=axs[1])
axs[1].set_ylabel("Tests [#]")
axs[1].legend_.remove()


for i, ax in enumerate(axs):
    ax = plt.gca()
    axs[i].xaxis.set_major_formatter(mdates.DateFormatter('%y-%m'))
    #axs[i].xaxis.set_major_locator(mdates.MonthLocator(bymonth=(1, 3, 5, 7, 9)))

plt.tight_layout()

## Wasewater data

In [ ]:
df_ww = pd.read_csv("https://static-content.springer.com/esm/art%3A10.1038%2Fs41598-024-64864-1/MediaObjects/41598_2024_64864_MOESM2_ESM.csv", sep=";")

In [ ]:
df_ww["Datum der Entnahme"] = pd.to_datetime(df_ww["Datum der Entnahme"])
df_ww["Name der Kläranlage"] = df_ww["Name der Kläranlage"].str.strip()
df_ww["Name der Kläranlage"] = df_ww["Name der Kläranlage"].str.replace("BASF ", "")
df_ww = df_ww.loc[df_ww["Name der Kläranlage"].isin(df["Town"].unique())]
df_ww = df_ww.loc[df_ww["Probe Valide"].str.strip()=="ja"]
df_ww = df_ww.loc[:, ['ProbeNr', 'Datum der Entnahme', "Name der Kläranlage", 'Probenbegleitschein vollständig', 'Genkopien im Durchschnitt (Gq/ml)',
       'Genkopien N1/ml', 'Genkopien N2/ml', 'Anteil Genkopien Durchschnitt zu PMMoV x100.000', 'Volumenstrom (m3/d)']]

In [ ]:
df_ww["Anteil Genkopien Durchschnitt zu PMMoV x100.000"] = df_ww["Anteil Genkopien Durchschnitt zu PMMoV x100.000"].astype(float)
df_ww["Genkopien im Durchschnitt (Gq/ml)"] = df_ww["Genkopien im Durchschnitt (Gq/ml)"].astype(float)
df_ww["Genkopien N1/ml"] = df_ww["Genkopien N1/ml"].astype(float)
df_ww["Genkopien N2/ml"] = df_ww["Genkopien N2/ml"].astype(float)
df_ww.sort_values("Name der Kläranlage", inplace=True)

In [ ]:
df_ww["Volumenstrom (m3/d)"] = df_ww["Volumenstrom (m3/d)"].str.replace("NA", "")
df_ww["Volumenstrom (m3/d)"] = df_ww["Volumenstrom (m3/d)"].str.replace("nicht messbar", "")

In [ ]:
df_ww["Volumenstrom (m3/d)"] = pd.to_numeric(df_ww["Volumenstrom (m3/d)"], errors='coerce')

In [ ]:
df_ww.head()

In [ ]:
df_ww.describe()

In [ ]:
fig, axs = plt.subplots(nrows=2, figsize=(9, 5), dpi=300, sharex=True)
sns.lineplot(data=df_ww, x="Datum der Entnahme", y="Anteil Genkopien Durchschnitt zu PMMoV x100.000", hue="Name der Kläranlage", ax=axs[0], legend=False)
axs[0].set_ylabel("Log Viral load\n(PMMoV normalized x 1e5)")
axs[0].set_yscale("log")
sns.lineplot(data=df_sub, x="Date of SARS-CoV-2 test", y="prevalence", hue="Town", ax=axs[1])
sns.scatterplot(data=df_sub, x="Date of SARS-CoV-2 test", y="prevalence", hue="Town", ax=axs[1], legend=False)
axs[1].set_ylabel("Prevalence [%]")


for i, ax in enumerate(axs):
    ax = plt.gca()
    axs[i].xaxis.set_major_formatter(mdates.DateFormatter('%y-%m'))
    #axs[i].xaxis.set_major_locator(mdates.MonthLocator(bymonth=(1, 3, 5, 7, 9)))

plt.tight_layout()

## Officially reported cases

In [ ]:
city_mapper = {
    "Kaiserslautern": 7335, # kreisfreie Stadt, 
    "Koblenz": 7111, # kreisfreie Stadt
    "Ludwigshafen": 7314, #kreisfreie Stadt
    "Mainz": 7315, # kreisfreie Stadt
    "Trier": 7211, # kreisfreie Stadt
}

In [ ]:
url = "https://raw.githubusercontent.com/robert-koch-institut/COVID-19_7-Tage-Inzidenz_in_Deutschland/main/COVID-19-Faelle_7-Tage-Inzidenz_Landkreise.csv"
df_cases = pd.read_csv(url)

In [ ]:
df_cases["Meldedatum"] = pd.to_datetime(df_cases["Meldedatum"])
df_cases = df_cases.loc[df_cases["Meldedatum"]>=df_ww["Datum der Entnahme"].min()]
df_cases = df_cases.loc[df_cases["Meldedatum"]<=df_ww["Datum der Entnahme"].max()]
df_cases = df_cases.loc[df_cases["Landkreis_id"].isin(city_mapper.values()), :] 
df_cases["Town"] = df_cases["Landkreis_id"].map({v: k for k, v in city_mapper.items()})

df_cases.sort_values(["Town"], inplace=True)

In [ ]:
df_cases.head()

In [ ]:
fig, axs = plt.subplots(nrows=3, figsize=(9, 7), dpi=300, sharex=True)
sns.lineplot(data=df_ww, x="Datum der Entnahme", y="Anteil Genkopien Durchschnitt zu PMMoV x100.000", hue="Name der Kläranlage", ax=axs[0], legend=False)
axs[0].set_ylabel("Viral load\n(PMMoV normalized x 1e5)")
sns.lineplot(data=df_sub, x="Date of SARS-CoV-2 test", y="prevalence", hue="Town", ax=axs[1])
sns.scatterplot(data=df_sub, x="Date of SARS-CoV-2 test", y="prevalence", hue="Town", ax=axs[1], legend=False)
axs[1].set_ylabel("Prevalence [%]")

sns.lineplot(data=df_cases, x="Meldedatum", y="Faelle_7-Tage", hue="Town", ax=axs[2], legend=False)
axs[2].set_ylabel("7-day case count [#] ")
axs[2].set_xlabel("Date")

for i, ax in enumerate(axs):
    ax = plt.gca()
    axs[i].xaxis.set_major_formatter(mdates.DateFormatter('%y-%m'))
    #axs[i].xaxis.set_major_locator(mdates.MonthLocator(bymonth=(1, 3, 5, 7, 9)))

plt.tight_layout()

In [ ]:

fig, axs = plt.subplots(nrows=3, figsize=(9, 7), dpi=300, sharex=True)
sns.lineplot(data=df_ww, x="Datum der Entnahme", y="Anteil Genkopien Durchschnitt zu PMMoV x100.000", hue="Name der Kläranlage", ax=axs[0], legend=False)
axs[0].set_ylabel("Viral load\n(PMMoV normalized x 1e5)")
sns.lineplot(data=df_sub, x="Date of SARS-CoV-2 test", y="prevalence", hue="Town", ax=axs[1])
sns.scatterplot(data=df_sub, x="Date of SARS-CoV-2 test", y="prevalence", hue="Town", ax=axs[1], legend=False)
axs[1].set_ylabel("Prevalence [%]")

sns.lineplot(data=df_cases, x="Meldedatum", y="Faelle_7-Tage", hue="Town", ax=axs[2], legend=False)
axs[2].set_ylabel("7-day case count [#] ")
axs[2].set_xlabel("Date")

for i, ax in enumerate(axs):
    ax = plt.gca()
    axs[i].xaxis.set_major_formatter(mdates.DateFormatter('%y-%m'))
    #axs[i].xaxis.set_major_locator(mdates.MonthLocator(bymonth=(1, 3, 5, 7, 9)))

plt.ylim(0, 200)
plt.tight_layout()

### Merge into one dataframe

In [ ]:
df_all = df_ww.rename(columns={"Datum der Entnahme": "Date", "Name der Kläranlage": "Town"}).merge(df_cases.rename(columns={"Meldedatum": "Date"}), on=["Date", "Town"], how="outer").drop(columns=[ "ProbeNr", "Landkreis_id"])
df_all = df_all.merge(df_sub.rename(columns={"Date of SARS-CoV-2 test": "Date"}), on=["Date", "Town"], how="outer")
df_all = df_all.loc[df_all.Date <= df_ww["Datum der Entnahme"].max()]

In [ ]:
df_all.head()

In [ ]:
df_all.to_csv("../data/preprocessed/rhineland_palatinate_data.csv", index=False)

In [ ]:
df_all.groupby("Town")["Inzidenz_7-Tage"].describe()

In [ ]:
df_all.Town.unique()